# 2일차 7교시 실습 · 색상 객체 추적
강의자료 PDF 1~16페이지. 빈칸은 A·B 두 개입니다. data 폴더와 tracking_helpers.py를 같은 폴더에 둡니다. 영상 입력·후보 계산·상태 관리·경로 표시는 준비되어 있습니다.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
from collections import Counter
from tracking_helpers import candidates, choose, overlay, read_image, show, track_video
data = Path("data")

## 거리와 연결 조건 · 강의자료 PDF 5·7페이지
A: NumPy 거리 계산 함수. B: 허용 거리와 같을 때도 연결하는 비교 연산자입니다. 아래 좌표는 설명용 값입니다.


In [ ]:
def point_distance(previous, current):
    px, py = previous
    cx, cy = current
    return float(np.____(cx - px, cy - py))  # A: PPT 5

def within_limit(distance, max_distance):
    return distance ____ max_distance  # B: PPT 7

p = (100.0, 80.0)
print(point_distance(p, (106.0, 88.0)))
print(point_distance(p, (124.0, 98.0)))
print([within_limit(d, 20) for d in [10,20,30]])

## 대표 위치 · 강의자료 PDF 3페이지
박스 중심을 float 좌표로 계산하며 화면 표시 때만 반올림합니다. 녹색 공 영상은 원본을 576×320으로 축소한 좌표계입니다.


In [ ]:
frame = read_image(data / "green50.png")
found, mask = candidates(frame)
print(found[0])
show([overlay(frame, found), mask], ["Candidates", "Mask"])

## 가까운 후보 강의자료 PDF 4·6페이지
노란 공 영상은 별도 HSV 범위를 적용한 실제 후보입니다. 이전 기준은 프레임 103의 오른쪽 후보를 지정합니다. 자동 초기화 결과가 아닙니다. 함수 내부 목록은 0부터, 화면 후보 번호는 1부터 시작합니다.


In [ ]:
before = read_image(data / "yellow103.png")
now = read_image(data / "yellow104.png")
old, _ = candidates(before, "yellow")
found_yellow, _ = candidates(now, "yellow")
previous = old[1]["xy"]
distances = [point_distance(previous, c["xy"]) for c in found_yellow]
selected, status, distance = choose(found_yellow, previous, point_distance, within_limit, 35)
print(distances, "Selected:", selected + 1, status)
show([overlay(before,old,1,manual=True), overlay(now,found_yellow,selected,previous=previous)], ["Previous 103", "Current 104"])

## 영상에 적용 · 강의자료 PDF 8·9페이지
normal.avi는 원본 프레임 20~140, 총 121프레임입니다. 첫 후보가 하나일 때 시작합니다. 기록의 frame은 포함 클립에서 0부터 시작하는 번호이며, 원본 번호는 frame+20입니다. 최근 30개 위치를 표시합니다. outputs/normal.avi와 마지막 프레임 PNG를 저장합니다.


In [ ]:
records = track_video(
    data / "normal.avi", point_distance, within_limit,
    max_distance=35, output="outputs/normal.avi")
print(Counter(r["status"] for r in records))
show([read_image("outputs/normal.png")], ["Normal segment"])

## 허용 거리 비교 · 강의자료 PDF 10페이지
영상·후보 조건은 그대로 두고 허용 거리만 8로 바꿉니다. start는 첫 시작 또는 새 경로 시작이며 연결 성공 횟수와 구분합니다. too_far는 후보가 있으나 허용 거리를 넘은 상태입니다.


In [ ]:
small_limit_records = track_video(
    data / "normal.avi", point_distance, within_limit,
    max_distance=8, output="outputs/limit8.avi")
print(Counter(r["status"] for r in small_limit_records))

## 누락과 재시작 · 강의자료 PDF 11페이지
gap.avi는 원본 220~245입니다. 후보가 없으면 이전 위치와 현재 경로를 비웁니다. 다음 프레임부터 후보 하나가 확인되면 새 구간을 시작합니다. 같은 물체를 재식별했다는 뜻은 아닙니다.


In [ ]:
gap_records = track_video(
    data / "gap.avi", point_distance, within_limit,
    max_distance=35, output="outputs/gap.avi")
print([(r["frame"]+220, r["status"], r["segment"]) for r in gap_records if r["frame"]+220 in [229,233,237]])

## 다른 공으로 연결되는 사례 · 강의자료 PDF 12페이지
프레임 127의 아래 공을 이전 기준으로 지정하고, 130과 비교합니다. 이 비교는 3프레임 간격이며 허용 거리는 120 pixel입니다. 128·129는 실제 이동을 확인하는 참고 프레임입니다. 자동 추적 정답 라벨을 사용한 사례가 아닙니다.


In [ ]:
images = [read_image(data / f"yellow{i}.png") for i in [127,128,129,130]]
old_cross, _ = candidates(images[0], "yellow")
now_cross, _ = candidates(images[-1], "yellow")
previous_cross = old_cross[0]["xy"]
cross_distances = [point_distance(previous_cross,c["xy"]) for c in now_cross]
cross_selected, cross_status, _ = choose(now_cross,previous_cross,point_distance,within_limit,120)
print(cross_distances, "Selected:", cross_selected+1)
show(images, ["127", "128", "129", "130"])
show([overlay(images[-1],now_cross,cross_selected,previous=previous_cross,distance_lines=True)], ["127 to 130 association"])

## 선택 방식 비교 · 강의자료 PDF 13페이지
4·6페이지와 동일한 후보 목록과 이전 기준을 사용합니다. 한 방식은 면적 최대, 다른 방식은 직전 위치와의 거리를 사용합니다.


In [ ]:
largest = int(np.argmax([c["area"] for c in found_yellow]))
nearest = int(np.argmin(distances))
print("Largest:", largest+1, "Nearest:", nearest+1)
show([overlay(now,found_yellow,largest), overlay(now,found_yellow,nearest,previous=previous)], ["Largest area", "Nearest position"])

## 다른 구간 적용 · 강의자료 PDF 14페이지
같은 원본의 700~780 구간입니다. 독립 촬영 검증이 아닙니다. 기본 35 pixel을 그대로 적용합니다.


In [ ]:
other_records = track_video(
    data / "other.avi", point_distance, within_limit,
    max_distance=35, output="outputs/other.avi")
print(Counter(r["status"] for r in other_records))
print([(r["frame"]+700,r["distance"]) for r in other_records if r["status"]=="too_far"])

## 위치 변화 비교 · 강의자료 PDF 15페이지
같은 크기의 영상에서 1프레임 간격의 두 구간을 비교합니다. 계산값은 화면 좌표의 거리입니다.


In [ ]:
move_distances=[]
for before_id, now_id in [(29,30),(223,224)]:
    f0=read_image(data / f"green{before_id}.png")
    f1=read_image(data / f"green{now_id}.png")
    c0,_=candidates(f0)
    c1,_=candidates(f1)
    d=point_distance(c0[0]["xy"],c1[0]["xy"])
    move_distances.append(d)
    print(before_id,now_id,d,within_limit(d,35))
    show([overlay(f0,c0),overlay(f1,c1)],[str(before_id),str(now_id)])

## 작은 좌표 변화 · 강의자료 PDF 16페이지
실제 프레임 75~77을 비교합니다. 완전한 정지 상태를 검증한 실험은 아닙니다. 좌표 변화만으로 물체 이동과 검출 영역 변화의 영향을 분리하지 않습니다.


In [ ]:
held_images=[]
for number in [75,76,77]:
    f=read_image(data / f"green{number}.png")
    cs,_=candidates(f)
    print(number,cs[0]["xy"],cs[0]["box"])
    held_images.append(overlay(f,cs))
show(held_images,["75","76","77"])